# CPV Statistical Validation for Journal of Pharmaceutical Innovation

**Purpose.** Reproducible Monte Carlo evaluation of prospective Nelson-rule strategies and offline single-change localization for batch-level Continued Process Verification (CPV).

This notebook is a manuscript extension to the synthetic CPV project. It does **not** contain proprietary manufacturing data. The design follows an ADEMP-style structure: aims, data-generating mechanisms, estimands, methods, and performance measures.

Key distinction:
- Nelson-rule strategies are evaluated as **prospective detection rules**.
- Change-point analysis is evaluated as an **offline post-signal localization tool**, not as a causal estimator or real-time alarm.


## Study design

- 20,000 Monte Carlo replicates per scenario.
- 30 monitoring observations per replicate.
- For changed scenarios, observations 1–10 are in control and the change begins at observation 11.
- Prospective strategies: Rule 1 only; Rules 1–3; Rules 1–8.
- Primary benchmark assumes the in-control mean and standard deviation are known, isolating rule behavior.
- Sensitivity analysis estimates the Phase I mean and sample SD from 18, 30, 50, or 100 in-control observations.
- Robustness analyses assess heavy-tailed data and positive autocorrelation.
- Detection delay is 0 when the first signal occurs on the first changed observation.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from cpv_jpi_simulation import (
    SEED, N_SIM, N_MON, PRE_CHANGE, RULE_SETS, SCENARIOS,
    run_unit_tests, scenario_design_table, primary_oracle_results,
    phase1_sensitivity_results, phase1_n18_detection_results,
    stress_test_results, horizon_sensitivity_results, change_point_results, make_figures
)

ROOT = Path.cwd()
print('Working directory:', ROOT)
print('Monte Carlo replicates per scenario:', N_SIM)
print('Monitoring observations:', N_MON)
print('Change starts at observation:', PRE_CHANGE + 1)


Working directory: /mnt/data/JPI_CPV_Paper_Package
Monte Carlo replicates per scenario: 20000
Monitoring observations: 30
Change starts at observation: 11


## Code validation

In [2]:
run_unit_tests()
print('All eight Nelson-rule unit tests and the change-point sanity check passed.')

All eight Nelson-rule unit tests and the change-point sanity check passed.


## Table 1. Prespecified simulation scenarios

In [3]:
design = scenario_design_table()
design

,Scenario,Data-generating mechanism,Change timing,Primary purpose
0,Stable,"IID N(0,1) throughout",No change,False-signal probability
1,Abrupt +1 SD,Mean shifts from 0 to +1 SD after observation 10,Boundary 10/11,Small sustained shift
2,Abrupt +2 SD,Mean shifts from 0 to +2 SD after observation 10,Boundary 10/11,Moderate sustained shift
3,Abrupt +3 SD,Mean shifts from 0 to +3 SD after observation 10,Boundary 10/11,Large sustained shift
4,Gradual drift to +3 SD,Linear mean drift from 0 to +3 SD over observa...,No single true boundary,Trend sensitivity
5,SD x2,SD doubles from 1 to 2 after observation 10; m...,Boundary 10/11,Variance instability


## Primary benchmark: prospective detection performance

In [4]:
primary = primary_oracle_results()
primary.round(3)

,Scenario,Method,N,Prechange_false_signal_pct,Eligible_no_prechange_N,Detection_probability_pct,Detection_CI_low_pct,Detection_CI_high_pct,Median_detection_delay,Mean_detection_delay
0,Stable,Rule 1 only,20000,NaN,NaN,7.850,7.485,8.231,NaN,NaN
1,Stable,Rules 1-3,20000,NaN,NaN,16.940,16.427,17.466,NaN,NaN
2,Stable,Rules 1-8,20000,NaN,NaN,30.900,30.263,31.544,NaN,NaN
3,Abrupt +1 SD,Rule 1 only,20000,2.800,19440.0,37.042,36.366,37.724,8.0,8.716
4,Abrupt +1 SD,Rules 1-3,20000,4.455,19109.0,75.509,74.894,76.113,8.0,8.842
5,Abrupt +1 SD,Rules 1-8,20000,7.850,18430.0,93.831,93.474,94.169,6.0,6.645
6,Abrupt +2 SD,Rule 1 only,20000,2.755,19449.0,96.853,96.599,97.090,3.0,4.696
7,Abrupt +2 SD,Rules 1-3,20000,4.425,19115.0,99.974,99.939,99.989,3.0,3.777
8,Abrupt +2 SD,Rules 1-8,20000,7.850,18430.0,100.000,99.979,100.000,2.0,1.953
9,Abrupt +3 SD,Rule 1 only,20000,2.610,19478.0,100.000,99.980,100.000,0.0,0.997


### Interpretation guardrail
Detection probabilities for changed scenarios are conditional on no signal during the 10-observation pre-change period. This separates pre-change false signaling from post-change detection. Stable-process results report the probability of at least one false signal anywhere over the 30-observation monitoring horizon.

## Phase I reference-sample sensitivity

In [5]:
phase1 = phase1_sensitivity_results()
phase1.round(3)

,Phase_I_reference,Phase_I_n,Method,N,False_alarm_probability_pct,CI_low_pct,CI_high_pct
0,Estimated n=18,18.0,Rule 1 only,20000,19.865,19.318,20.424
1,Estimated n=18,18.0,Rules 1-3,20000,30.905,30.268,31.549
2,Estimated n=18,18.0,Rules 1-8,20000,47.715,47.023,48.408
3,Estimated n=30,30.0,Rule 1 only,20000,15.485,14.990,15.993
4,Estimated n=30,30.0,Rules 1-3,20000,26.150,25.546,26.764
5,Estimated n=30,30.0,Rules 1-8,20000,42.285,41.602,42.971
6,Estimated n=50,50.0,Rule 1 only,20000,11.960,11.518,12.417
7,Estimated n=50,50.0,Rules 1-3,20000,21.790,21.223,22.368
8,Estimated n=50,50.0,Rules 1-8,20000,37.740,37.071,38.414
9,Estimated n=100,100.0,Rule 1 only,20000,9.900,9.494,10.322


## Detection performance when Phase I n = 18

In [6]:
n18_detection = phase1_n18_detection_results()
n18_detection.round(3)

,Scenario,Method,N,Prechange_false_signal_pct,Eligible_no_prechange_N,Detection_probability_pct,Detection_CI_low_pct,Detection_CI_high_pct,Median_detection_delay,Mean_detection_delay
0,Abrupt +1 SD,Rule 1 only,20000,8.325,18335,41.849,41.137,42.565,7.0,7.405
1,Abrupt +1 SD,Rules 1-3,20000,10.775,17845,74.463,73.818,75.098,8.0,7.882
2,Abrupt +1 SD,Rules 1-8,20000,16.940,16612,88.725,88.235,89.197,5.0,6.135
3,Abrupt +2 SD,Rule 1 only,20000,8.145,18371,87.420,86.933,87.892,3.0,4.300
4,Abrupt +2 SD,Rules 1-3,20000,10.640,17872,99.603,99.499,99.685,3.0,3.889
5,Abrupt +2 SD,Rules 1-8,20000,17.040,16592,99.982,99.947,99.994,2.0,2.117
6,Abrupt +3 SD,Rule 1 only,20000,8.315,18337,99.389,99.266,99.492,0.0,1.372
7,Abrupt +3 SD,Rules 1-3,20000,10.640,17872,100.000,99.979,100.000,0.0,1.242
8,Abrupt +3 SD,Rules 1-8,20000,16.990,16602,100.000,99.977,100.000,0.0,0.718
9,Gradual drift to +3 SD,Rule 1 only,20000,8.375,18325,89.097,88.637,89.540,13.0,12.082


## Robustness stress tests

In [7]:
stress = stress_test_results()
stress.round(3)

,Stress_condition,Method,N,False_alarm_probability_pct,CI_low_pct,CI_high_pct
0,Normal IID (true parameters),Rule 1 only,20000,7.970,7.603,8.353
1,Normal IID (true parameters),Rules 1-3,20000,17.205,16.688,17.734
2,Normal IID (true parameters),Rules 1-8,20000,30.815,30.179,31.459
3,"Student t(5), unit variance",Rule 1 only,20000,30.155,29.523,30.795
4,"Student t(5), unit variance",Rules 1-3,20000,37.000,36.333,37.672
5,"Student t(5), unit variance",Rules 1-8,20000,47.270,46.579,47.962
6,"AR(1), rho=0.3, unit variance",Rule 1 only,20000,7.165,6.816,7.531
7,"AR(1), rho=0.3, unit variance",Rules 1-3,20000,31.525,30.885,32.172
8,"AR(1), rho=0.3, unit variance",Rules 1-8,20000,52.115,51.422,52.807
9,"Normal IID, estimated Phase I n=18",Rule 1 only,20000,19.615,19.071,20.171


## Monitoring-horizon sensitivity

In [8]:
horizon = horizon_sensitivity_results()
horizon.round(3)

,Monitoring_horizon_batches,Method,N,False_alarm_probability_pct,CI_low_pct,CI_high_pct
0,15,Rule 1 only,20000,3.725,3.471,3.996
1,15,Rules 1-3,20000,7.570,7.211,7.945
2,15,Rules 1-8,20000,14.095,13.620,14.584
3,30,Rule 1 only,20000,7.805,7.441,8.185
4,30,Rules 1-3,20000,17.190,16.673,17.719
5,30,Rules 1-8,20000,31.155,30.517,31.800
6,60,Rule 1 only,20000,15.085,14.596,15.588
7,60,Rules 1-3,20000,32.655,32.008,33.308
8,60,Rules 1-8,20000,54.905,54.215,55.594


## Offline change-point localization

In [9]:
cp = change_point_results()
cp.round(3)

,Scenario,N,True_boundary,Mean_estimated_boundary,Median_estimated_boundary,MAE_batches,Median_abs_error_batches,Exact_pct,Within_1_batch_pct,Within_2_batches_pct
0,Abrupt +1 SD,20000,10.0,11.369,10.0,2.901,2.0,26.725,48.815,62.330
1,Abrupt +2 SD,20000,10.0,10.142,10.0,0.737,0.0,61.860,83.850,91.625
2,Abrupt +3 SD,20000,10.0,10.025,10.0,0.207,0.0,84.005,96.770,99.050
3,Gradual drift to +3 SD,20000,NaN,18.111,18.0,NaN,NaN,NaN,NaN,NaN


For abrupt shifts, localization error is measured against the known boundary between monitoring observations 10 and 11. For gradual drift, no single 'true' change boundary exists; the reported split is therefore descriptive of the dominant least-squares segmentation and must not be interpreted as the onset of drift.

## Generate publication figures and machine-readable outputs

In [10]:
make_figures(primary, phase1, cp)
print('Figures written to:', ROOT / 'figures')
print('Existing result tables are in:', ROOT / 'results')

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Figures written to: /mnt/data/JPI_CPV_Paper_Package/figures
Existing result tables are in: /mnt/data/JPI_CPV_Paper_Package/results


## Reproducibility notes

1. Random-number seeds are fixed in `cpv_jpi_simulation.py`.
2. The code defines the rule logic explicitly rather than relying on a black-box SPC package.
3. Each Nelson rule has a deterministic unit test.
4. Monte Carlo 95% confidence intervals for proportions use the Wilson method.
5. The simulation evaluates statistical operating characteristics under stated data-generating assumptions; it does not validate a GMP production system.
6. Before manuscript submission, the author should rerun this notebook in the archived repository environment and confirm that the generated tables and figures match the manuscript exactly.
